In [4]:
import requests
import pandas as pd
from mainnet_launch.constants import AUTO_ETH
from web3 import Web3


def paginate_query(api_url: str, query: str, variables: dict, data_col: str) -> list[dict]:
    """
    Helper to page through a GraphQL connection using `first`/`skip`.

    :param api_url: GraphQL endpoint
    :param query: The GraphQL query string, expecting $first and $skip variables
    :param variables: Base variables (e.g. {"autoEthAddress": "..."}).
                      first/skip will be merged in each loop.
    :param batch_size: Number of items to fetch per request.
    :return: List of result dicts
    """

    all_records = []
    skip = 0

    while True:
        vars_with_pagination = {**variables, "first": 500, "skip": skip}
        resp = requests.post(api_url, json={"query": query, "variables": vars_with_pagination})
        resp.raise_for_status()
        batch = resp.json()["data"][data_col]

        if not batch:
            break

        all_records.extend(batch)
        skip += 500

    return all_records


def _get_subgraph_api(chain):
    if chain == "eth":
        api_url = "https://subgraph.satsuma-prod.com/108d48ba91e3/tokemak/v2-gen3-eth-mainnet/api"
    elif chain == "base":
        api_url = "https://subgraph.satsuma-prod.com/108d48ba91e3/tokemak/v2-gen3-base-mainnet/api"
    else:
        raise ValueError("bad chain", chain)

    return api_url


def fetch_autopool_rebalance_events_from_subgraph(autopool_eth_addr: str, chain: str) -> list[dict]:
    """
    Fetches all AutopoolRebalances entries for the given autopool.
    """
    subgraph_url = _get_subgraph_api(chain)

    query = """
    query($autoEthAddress: String!, $first: Int!, $skip: Int!) {
      autopoolRebalances(
        first: $first,
        skip: $skip,
        orderBy: id,
        orderDirection: desc,
        where: { autopool: $autoEthAddress }
      ) {
        transactionHash
        timestamp
        blockNumber

        tokenInAmount
        tokenOut {
        id
        decimals
        
        }        
        destinationInAddress

        tokenIn {
        id
        decimals
        
        }   

        tokenOutAmount
        destinationOutAddress
        
      }
    }
    """

    # Adjust `first` as needed; paginate_query will loop over skip increments of `first`
    df = pd.DataFrame.from_records(
        paginate_query(
            subgraph_url,
            query,
            variables={"autoEthAddress": autopool_eth_addr.lower(), "first": 1000, "skip": 0},
            data_col="autopoolRebalances",
        )
    )

    df["tokenInAddress"] = df["tokenIn"].apply(lambda x: x["id"])
    df["tokenOutAddress"] = df["tokenOut"].apply(lambda x: x["id"])

    df["tokenOutAmount"] = df.apply(
        lambda row: int(row["tokenOutAmount"]) / (10 ** int(row["tokenOut"]["decimals"])), axis=1
    )
    df["tokenInAmount"] = df.apply(
        lambda row: int(row["tokenInAmount"]) / (10 ** int(row["tokenIn"]["decimals"])), axis=1
    )

    df["blockNumber"] = df["blockNumber"].astype(int)

    df["destinationInAddress"] = df["destinationInAddress"].apply(lambda x: Web3.toChecksumAddress(x))
    df["tokenInAddress"] = df["tokenInAddress"].apply(lambda x: Web3.toChecksumAddress(x))

    df["destinationOutAddress"] = df["destinationOutAddress"].apply(lambda x: Web3.toChecksumAddress(x))
    df["tokenOutAddress"] = df["tokenOutAddress"].apply(lambda x: Web3.toChecksumAddress(x))

    return df


rebalance_event_df = fetch_autopool_rebalance_events_from_subgraph("0xa7569A44f348d3D70d8ad5889e50F78E33d80D35", "eth")
rebalance_event_df

,transactionHash,timestamp,blockNumber,tokenInAmount,tokenOut,destinationInAddress,tokenIn,tokenOutAmount,destinationOutAddress,tokenInAddress,tokenOutAddress
0,0xff8915d0bd9054e7678c4366ae8fd5a0a3cce80943be...,1744988327,22296632,148881.150087,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
1,0xff3d22a3564c8121f28d7d47feffb8251785ad2c48e4...,1744997471,22297389,148885.474474,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
2,0xfecf659a8d1c06c45010afc47fcefc6405d906a85e84...,1746235043,22399927,320651.575857,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xC5c95fCad37E466E25e6ecA1977bbF75C0E1004a,{'id': '0xbeef01735c132ada46aa9aa4c54623caa92a...,347222.682150,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
3,0xfca3e4ad4e6296b2444ed28a8012cda8694ef5d23ce9...,1746116255,22390138,148075.994800,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7876F91BB22148345b3De16af9448081E9853830,{'id': '0x9fb7b4477576fe5b32be4c1843afb1e55f25...,170000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x9Fb7b4477576Fe5B32be4C1843aFB1e55F251B33,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
4,0xfc37985d3c7182369be30f9dbe69409a8b05d008bde2...,1745111003,22306807,437134.294459,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xE4545f9dBC30Ccb6Cda6930DDFd69f3D419FcB61,{'id': '0x5c20b550819128074fd538edf79791733cce...,500000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x5C20B550819128074FD538Edf79791733ccEdd18,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
...,...,...,...,...,...,...,...,...,...,...,...
264,0x02a5528289efe28718bed54e9c5b0772f16eba69d609...,1744695875,22272381,76347.504659,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,77679.210200,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
265,0x02065d1b16c7c693afe6358f38647b921a095d6876ec...,1745911055,22373171,26611.724622,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,27068.157900,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
266,0x012e26a2633c10388ffc3ee1a2a95c9569b54ff4f918...,1744556327,22260800,140053.102440,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,0xC099899d0278CE83976218Cbe58D01dD382dcA32,{'id': '0x57064f49ad7123c92560882a45518374ad98...,149036.479162,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,0x57064F49Ad7123C92560882a45518374ad982e85,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85
267,0x00ff23899cd9bb0075ebe5ac23a3faa815088ddd32ab...,1744906595,22289850,148973.430940,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48


In [5]:
from mainnet_launch.database.schema.full import Blocks, DestinationTokenValues, DestinationStates, Destinations
from mainnet_launch.database.schema.postgres_operations import (
    merge_tables_as_df,
    TableSelector,
)
from mainnet_launch.database.schema.ensure_tables_are_current.update_destinations_states_table import (
    _add_new_destination_states_to_db,
)

_add_new_destination_states_to_db(rebalance_event_df["blockNumber"], AUTO_ETH.chain)

destination_states_df = merge_tables_as_df(
    [
        TableSelector(DestinationStates, [DestinationStates.block, DestinationStates.lp_token_spot_price]),
        TableSelector(
            Destinations,
            [Destinations.destination_vault_address, Destinations.underlying_symbol],
            join_on=(DestinationStates.chain_id == DestinationStates.chain_id)
            & (Destinations.destination_vault_address == DestinationStates.destination_vault_address),
        ),
    ]
)
destination_states_df

2025-05-05 14:17:08,444 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-05-05 14:17:08,445 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM destination_states
            WHERE destination_states.chain_id = 1
        
2025-05-05 14:17:08,445 INFO sqlalchemy.engine.Engine [cached since 54.19s ago] {}
2025-05-05 14:17:10,584 INFO sqlalchemy.engine.Engine COMMIT
2025-05-05 14:17:10,724 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-05-05 14:17:10,725 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM blocks
            WHERE blocks.chain_id = 1
        
2025-05-05 14:17:10,725 INFO sqlalchemy.engine.Engine [cached since 54.09s ago] {}
2025-05-05 14:17:10,931 INFO sqlalchemy.engine.Engine COMMIT
2025-05-05 14:17:11,038 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-05-05 14:17:11,039 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM destinations
            WHERE destinations.chain_id = 1
        
2025-05-05 14:17:11,03

,block,lp_token_spot_price,destination_vault_address,underlying_symbol
0,20759464,1.036516,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,B-rETH-STABLE
1,20766617,1.036121,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,B-rETH-STABLE
2,20773761,1.036456,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,B-rETH-STABLE
3,20780916,1.036477,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,B-rETH-STABLE
4,20788084,1.036502,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,B-rETH-STABLE
...,...,...,...,...
34902,22272381,1.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,autoUSD
34903,22373171,1.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,autoUSD
34904,22260800,1.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,autoUSD
34905,22289850,1.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,autoUSD


In [6]:
price_df = destination_states_df[["block", "destination_vault_address", "lp_token_spot_price"]]

df1 = rebalance_event_df.merge(
    price_df,
    left_on=["blockNumber", "destinationInAddress"],
    right_on=["block", "destination_vault_address"],
    how="left",
)
# rename & drop the merge‑helper cols
df1.rename(columns={"lp_token_spot_price": "lp_token_in_spot_price"}, inplace=True)
df1.drop(columns=["block", "destination_vault_address"], inplace=True)


df1 = df1.merge(
    price_df,
    left_on=["blockNumber", "destinationOutAddress"],
    right_on=["block", "destination_vault_address"],
    how="left",
)
df1.rename(columns={"lp_token_spot_price": "lp_token_out_spot_price"}, inplace=True)
df1.drop(columns=["block", "destination_vault_address"], inplace=True)


# result
df1["spot_value_in"] = df1["tokenInAmount"] * df1["lp_token_in_spot_price"]
df1["spot_value_out"] = df1["tokenOutAmount"] * df1["lp_token_out_spot_price"]
df1

,transactionHash,timestamp,blockNumber,tokenInAmount,tokenOut,destinationInAddress,tokenIn,tokenOutAmount,destinationOutAddress,tokenInAddress,tokenOutAddress,lp_token_in_spot_price,lp_token_out_spot_price,spot_value_in,spot_value_out
0,0xff8915d0bd9054e7678c4366ae8fd5a0a3cce80943be...,1744988327,22296632,148881.150087,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.006900,1.000000,149908.430023,150000.000000
1,0xff3d22a3564c8121f28d7d47feffb8251785ad2c48e4...,1744997471,22297389,148885.474474,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.006902,1.000000,149913.082019,150000.000000
2,0xfecf659a8d1c06c45010afc47fcefc6405d906a85e84...,1746235043,22399927,320651.575857,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xC5c95fCad37E466E25e6ecA1977bbF75C0E1004a,{'id': '0xbeef01735c132ada46aa9aa4c54623caa92a...,347222.682150,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.082884,1.000000,347228.461070,347222.682150
3,0xfca3e4ad4e6296b2444ed28a8012cda8694ef5d23ce9...,1746116255,22390138,148075.994800,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7876F91BB22148345b3De16af9448081E9853830,{'id': '0x9fb7b4477576fe5b32be4c1843afb1e55f25...,170000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x9Fb7b4477576Fe5B32be4C1843aFB1e55F251B33,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.148038,1.000000,169996.868918,170000.000000
4,0xfc37985d3c7182369be30f9dbe69409a8b05d008bde2...,1745111003,22306807,437134.294459,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xE4545f9dBC30Ccb6Cda6930DDFd69f3D419FcB61,{'id': '0x5c20b550819128074fd538edf79791733cce...,500000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x5C20B550819128074FD538Edf79791733ccEdd18,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.143221,1.000000,499741.105246,500000.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,0x02a5528289efe28718bed54e9c5b0772f16eba69d609...,1744695875,22272381,76347.504659,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,77679.210200,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.017084,1.000000,77651.825428,77679.210200
265,0x02065d1b16c7c693afe6358f38647b921a095d6876ec...,1745911055,22373171,26611.724622,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,27068.157900,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.017144,1.000000,27067.956028,27068.157900
266,0x012e26a2633c10388ffc3ee1a2a95c9569b54ff4f918...,1744556327,22260800,140053.102440,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,0xC099899d0278CE83976218Cbe58D01dD382dcA32,{'id': '0x57064f49ad7123c92560882a45518374ad98...,149036.479162,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,0x57064F49Ad7123C92560882a45518374ad982e85,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,1.071202,1.006465,150025.163440,150000.000000
267,0x00ff23899cd9bb0075ebe5ac23a3faa815088ddd32ab...,1744906595,22289850,148973.430940,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xbB2d2dd491204a86ec10a1a6972F940B34fE060e,{'id': '0x4f493b7de8aac7d55f71853688b1f7c8f024...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4f493B7dE8aAC7d55F71853688b1F7C8F0243C85,0xA0b86991c6218b36c1d19D4a2e

In [12]:
df1['slippage'] = (100 * (df1['spot_value_out'] - df1['spot_value_in'])) / df1['spot_value_out']

import plotly .express as px



df1.sort_values('slippage')

,transactionHash,timestamp,blockNumber,tokenInAmount,tokenOut,destinationInAddress,tokenIn,tokenOutAmount,destinationOutAddress,tokenInAddress,tokenOutAddress,lp_token_in_spot_price,lp_token_out_spot_price,spot_value_in,spot_value_out,slippage
239,0x1b154344809534e52c7b7293056c2ec63e61ebffcd2d...,1746330335,22407809,5474.504203,{'id': '0xbeef01735c132ada46aa9aa4c54623caa92a...,0x65efCF2cce562DCBf07e805eEbeDeF21Dbd8Ea3D,{'id': '0x4dece678ceceb27446b35c672dc7d61f30ba...,5021.109138,0xC5c95fCad37E466E25e6ecA1977bbF75C0E1004a,0x4DEcE678ceceb27446b35C672dC7d61F30bAD69E,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,1.018368,1.083027,5575.059896,5437.996766,-2.520471
150,0x6ab3f09e6a2ec4ef83c2dbd851f3c92a10f8fd85a611...,1745457407,22335534,148393.790366,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x65efCF2cce562DCBf07e805eEbeDeF21Dbd8Ea3D,{'id': '0x4dece678ceceb27446b35c672dc7d61f30ba...,150000.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x4DEcE678ceceb27446b35C672dC7d61F30bAD69E,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.018319,1.000000,151112.216211,150000.000000,-0.741477
24,0xeb1d3f3839174cf5b72ed6cd395bba5d40fd6e759169...,1745813843,22365119,147615.747990,{'id': '0x9fb7b4477576fe5b32be4c1843afb1e55f25...,0x65efCF2cce562DCBf07e805eEbeDeF21Dbd8Ea3D,{'id': '0x4dece678ceceb27446b35c672dc7d61f30ba...,130683.763587,0x7876F91BB22148345b3De16af9448081E9853830,0x4DEcE678ceceb27446b35C672dC7d61F30bAD69E,0x9Fb7b4477576Fe5B32be4C1843aFB1e55F251B33,1.018345,1.147809,150323.758887,149999.999999,-0.215839
65,0xb4ae4eee0be0b8522433086172131818c1d42868fb95...,1746181031,22395473,72565.917676,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,73720.267300,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.017142,1.000000,73809.842637,73720.267300,-0.121507
47,0xd0563c451348fa77eea46fa4b17aa32064713f6a0547...,1745897483,22372047,39892.996621,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x0f170e37E5E0C617148517B831A49292c38c363d,{'id': '0xdd0f28e19c1780eb6396170735d45153d261...,42760.785575,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0xdd0f28e19C1780eb6396170735D45153D261490d,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.072742,1.000000,42794.892981,42760.785575,-0.079763
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,0xb74d80cca9ff137d8f8812e3c93afae82ef593a44967...,1745982647,22379097,137401.041461,{'id': '0x5c20b550819128074fd538edf79791733cce...,0xCADe3658eCEA14e4da22ef81BE335E9783E68848,{'id': '0xb819feef8f0fcdc268afe14162983a69f6bf...,131075.342106,0xE4545f9dBC30Ccb6Cda6930DDFd69f3D419FcB61,0xb819feeF8F0fcDC268AfE14162983A69f6BF179E,0x5C20B550819128074FD538Edf79791733ccEdd18,1.090742,1.144380,149869.086765,149999.999999,0.087275
88,0x998a06402a2d42b4c5b23f2a9089378841c89833bace...,1745943479,22375858,42626.221887,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,43399.473650,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x390f3595bCa2Df7d23783dFd126427CCeb997BF4,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.017219,1.000000,43360.202802,43399.473650,0.090487
33,0xe59412dfb0dad3fbb1a9b9deecc20a1bed4e5da0ced6...,1745207099,22314769,134365.159686,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0xd7900d87069C815a299bdA7aFDcd7eEe98fe4b6c,{'id': '0x5b03cccab7ba3010fa5cad23746cbf079493...,135304.324490,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,0x5B03CcCAb7BA3010fA5CAd23746cbf0794938e96,0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48,1.006077,1.000000,135181.696762,135304.324490,0.090631
258,0x0619a8602841faf599a0b316e83f404fbf3a7bc0c3a1...,1744501463,22256246,172726.439342,{'id': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606...,0x7583b1589aDD33320366A48A92794D77763FAE9e,{'id': '0x390f3595bca2df7d23783dfd126427cceb99...,175803.150499,0xa7569A44f348d3D70d8ad5889e50F7

In [14]:
px.scatter(df1, x='spot_value_out', y='slippage')

In [7]:
destination_states_df

,block,lp_token_spot_price,destination_vault_address,underlying_symbol
0,20759464,1.036516,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,B-rETH-STABLE
1,20766617,1.036121,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,B-rETH-STABLE
2,20773761,1.036456,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,B-rETH-STABLE
3,20780916,1.036477,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,B-rETH-STABLE
4,20788084,1.036502,0xf3ae3c74EaD129e770A864CeE291A805b170bBe0,B-rETH-STABLE
...,...,...,...,...
34902,22272381,1.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,autoUSD
34903,22373171,1.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,autoUSD
34904,22260800,1.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,autoUSD
34905,22289850,1.000000,0xa7569A44f348d3D70d8ad5889e50F78E33d80D35,autoUSD


In [8]:
destination_states_df[destination_states_df["underlying"] == "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48"]

KeyError: 'underlying'

In [ ]:
df1["tokenOutAddress"].value_counts()

In [ ]:
destination_states_df

In [ ]:
rebalance_event_df

In [ ]:
rebalance_event_df[["blockNumber", "destinationInAddress", "tokenInAddress"]]

In [ ]:
destination_states_df

In [ ]:
destination_states_df[["block", "destination_vault_address", "underlying"]]

In [ ]:
destination_states_df = destination_states_df.drop_duplicates()

In [ ]:
rebalance_event_df.head(1).T

In [ ]:
# 0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48

In [ ]:
token_value_df[
    (token_value_df["block"] == 22296632)
    & (token_value_df["destination_vault_address"] == "0xa7569A44f348d3D70d8ad5889e50F78E33d80D35")
    & (token_value_df["token_address"] == "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48")
]

In [ ]:
rebalance_event_df

In [ ]:
destination_states_df["underlying_symbol"].value_counts()